In [1]:
import numpy as np
from sklearn.linear_model import LinearRegression
import pandas as pd

df = pd.read_csv('../../01-intro/1.9 Bike Sharing/data/day.csv')
df['dteday'] = pd.to_datetime(df['dteday'])
df["day_of_year"] = df["dteday"].dt.day_of_year
df = df.sort_values('dteday')
X_calendar = df[["season","yr","mnth","holiday","weekday","workingday", "day_of_year"]]


print(df[['dteday', 'day_of_year']].head())

y = df["cnt"]
X_train = X_calendar.iloc[:511]
X_val = X_calendar.iloc[511:511+109]
X_test = X_calendar.iloc[511+109:]
y_train = df["cnt"].iloc[:511]
y_val = df["cnt"].iloc[511:511+109]

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_val)



      dteday  day_of_year
0 2011-01-01            1
1 2011-01-02            2
2 2011-01-03            3
3 2011-01-04            4
4 2011-01-05            5


In [2]:
assert df.loc[
    df["dteday"] == "2011-03-01",
    "day_of_year"
].iloc[0] == 60

assert df.loc[
    df["dteday"] == "2012-03-01",
    "day_of_year"
].iloc[0] == 61

In [3]:
weekday_encoded = pd.get_dummies(
    df["weekday"],
    prefix="weekday",
    dtype=int
)

print(weekday_encoded.head())
assert weekday_encoded.sum(axis=1).eq(1).all()
assert weekday_encoded.loc[df["weekday"] == 6, "weekday_6"].eq(1).all()


   weekday_0  weekday_1  weekday_2  weekday_3  weekday_4  weekday_5  weekday_6
0          0          0          0          0          0          0          1
1          1          0          0          0          0          0          0
2          0          1          0          0          0          0          0
3          0          0          1          0          0          0          0
4          0          0          0          1          0          0          0


In [4]:
shuffled_encoded = weekday_encoded.sample(
    frac=1,
    random_state=42
)

print(shuffled_encoded.head())
print(shuffled_encoded.index[:5])

assert shuffled_encoded.loc[df["weekday"] == 6, "weekday_6"].eq(1).all()


     weekday_0  weekday_1  weekday_2  weekday_3  weekday_4  weekday_5  \
703          0          0          1          0          0          0   
33           0          0          0          0          1          0   
300          0          0          0          0          0          1   
456          1          0          0          0          0          0   
633          0          0          1          0          0          0   

     weekday_6  
703          0  
33           0  
300          0  
456          0  
633          0  
Index([703, 33, 300, 456, 633], dtype='int64')


In [5]:
df["day_sin"] = np.sin(
    2 * np.pi * df["day_of_year"] / 365
)

print(df[["dteday", "day_of_year", "day_sin"]].head())

      dteday  day_of_year   day_sin
0 2011-01-01            1  0.017213
1 2011-01-02            2  0.034422
2 2011-01-03            3  0.051620
3 2011-01-04            4  0.068802
4 2011-01-05            5  0.085965


In [6]:
df["day_cos"] = np.cos(
    2 * np.pi * df["day_of_year"] / 365
)

print(df[["dteday", "day_sin", "day_cos"]].head())

      dteday   day_sin   day_cos
0 2011-01-01  0.017213  0.999852
1 2011-01-02  0.034422  0.999407
2 2011-01-03  0.051620  0.998667
3 2011-01-04  0.068802  0.997630
4 2011-01-05  0.085965  0.996298


In [7]:
assert np.allclose(
    df["day_sin"] ** 2 + df["day_cos"] ** 2,
    1
)

In [8]:
df["days_in_year"] = np.where(df['dteday'].dt.is_leap_year, 366, 365)

print(df[["dteday", "day_of_year", "days_in_year"]].iloc[360:370])

        dteday  day_of_year  days_in_year
360 2011-12-27          361           365
361 2011-12-28          362           365
362 2011-12-29          363           365
363 2011-12-30          364           365
364 2011-12-31          365           365
365 2012-01-01            1           366
366 2012-01-02            2           366
367 2012-01-03            3           366
368 2012-01-04            4           366
369 2012-01-05            5           366


In [9]:
df["day_sin"] = np.sin(
    2 * np.pi * df["day_of_year"] / df["days_in_year"]
)

df["day_cos"] = np.cos(
    2 * np.pi * df["day_of_year"] / df["days_in_year"]
)

assert np.allclose(
    df["day_sin"] ** 2 + df["day_cos"] ** 2,
    1
)


In [10]:
dec_31 = df.loc[
    df["dteday"] == "2011-12-31",
    ["day_sin", "day_cos"]
].iloc[0]

In [11]:
jan_01 = df.loc[
    df["dteday"] == "2012-01-01",
    ["day_sin", "day_cos"]
].iloc[0]

In [12]:
assert np.allclose(dec_31, jan_01, atol=0.02)

In [13]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

#Original Split Dataset
X_original = df[["season","yr","mnth","holiday","weekday","workingday"]]
y = df["cnt"]
X_train = X_original.iloc[:511]
X_val = X_original.iloc[511:511+109]
X_test = X_original.iloc[511+109:]
y_train = df["cnt"].iloc[:511]
y_val = df["cnt"].iloc[511:511+109]

#Original Baseline
modelA = LinearRegression()
modelA.fit(X_train, y_train)
modelA_pred = modelA.predict(X_val)

#Original Error
modelA_mae = mean_absolute_error(y_val, modelA_pred)
modelA_rmse = root_mean_squared_error(y_val, modelA_pred)

print(modelA_mae, modelA_rmse)

1124.7784058910688 1300.4543998614956


In [14]:
#Engineered Split Dataset
X_engineered = df[["yr", "holiday", "workingday", "weekday", "season", "day_sin", "day_cos"]]
season_encoded = pd.get_dummies(X_engineered["season"], prefix="season", dtype=int)

assert season_encoded.sum(axis=1).eq(1).all()
assert season_encoded.loc[X_engineered["season"] == 1, "season_1"].eq(1).all()

season_encoded.head()


,season_1,season_2,season_3,season_4
0,1,0,0,0
1,1,0,0,0
2,1,0,0,0
3,1,0,0,0
4,1,0,0,0


In [15]:
reduced_season = season_encoded.drop(columns="season_1")
reduced_weekday = weekday_encoded.drop(columns="weekday_0")
reduced_engineered = X_engineered.drop(columns=["weekday", "season"])

X_modelB = pd.concat([reduced_season, reduced_weekday, reduced_engineered], axis=1)
print(X_modelB.shape)

X_etrain = X_modelB.iloc[:511]
X_eval = X_modelB.iloc[511:511+109]
X_etest = X_modelB.iloc[511+109:]

modelB = LinearRegression()
modelB.fit(X_etrain, y_train)
modelB_pred = modelB.predict(X_eval)

modelB_mae = mean_absolute_error(y_val, modelB_pred)
modelB_rmse = root_mean_squared_error(y_val, modelB_pred)

print(modelB_mae, modelB_rmse)

(731, 14)
711.2114922418989 932.9624429838236


In [16]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

modelC = make_pipeline(
    StandardScaler(), 
    Ridge(alpha=1.0)

)

print(modelC)

Pipeline(steps=[('standardscaler', StandardScaler()), ('ridge', Ridge())])


In [17]:
modelC.fit(X_etrain, y_train)
modelC_pred = modelC.predict(X_eval)

modelC_mae = mean_absolute_error(y_val, modelC_pred)
modelC_rmse = root_mean_squared_error(y_val, modelC_pred)

print(modelC_mae, modelC_rmse)


710.8693256261979 931.0977164564755


In [18]:
experiment_results = {
    "model": ["Model A", "Model B", "Model C"],
    "features": ["Original", "Engineered", "Engineered + Ridge"],
    "alpha": ["N/A", "N/A", 1.0],
    "mae": [modelA_mae, modelB_mae, modelC_mae],
    "rmse": [modelA_rmse, modelB_rmse, modelC_rmse]
}

results_df = pd.DataFrame(experiment_results).sort_values(by="mae", ascending=True).round(2).reset_index(drop=True)

results_df


,model,features,alpha,mae,rmse
0,Model C,Engineered + Ridge,1.0,710.87,931.10
1,Model B,Engineered,N/A,711.21,932.96
2,Model A,Original,N/A,1124.78,1300.45


In [19]:
modelA_bias = (y_val - modelA_pred).mean()
modelB_bias = (y_val - modelB_pred).mean()
modelC_bias = (y_val - modelC_pred).mean()

print(modelA_bias, modelB_bias, modelC_bias)

931.6766065706322 -51.71886727438257 -42.678748539430394


In [20]:
results_df["bias"] = [modelA_bias, modelB_bias, modelC_bias]

results_df

,model,features,alpha,mae,rmse,bias
0,Model C,Engineered + Ridge,1.0,710.87,931.10,931.676607
1,Model B,Engineered,N/A,711.21,932.96,-51.718867
2,Model A,Original,N/A,1124.78,1300.45,-42.678749


## Day 13 — Feature Engineering and Regularization

### Objective
Improve tomorrow's bike rental demand predictions through feature engineering and Ridge regularization, while preserving chronological validation and preventing data leakage.

### 1. Feature Engineering

**Categorical encoding:**
- One-hot encoded `weekday` and `season` to avoid imposing linear relationships on category IDs.
- Removed `weekday_0` and `season_1` as reference categories to avoid perfect multicollinearity with the intercept.

**Cyclical encoding:**
- Extracted `day_of_year` from `dteday`.
- Created `day_sin` and `day_cos` to represent annual seasonality.
- Accounted for leap years using 365 or 366 days per year.

**Final engineered features (14):**
- Numeric/binary: `yr`, `holiday`, `workingday`, `day_sin`, `day_cos`
- Six weekday indicators
- Three season indicators

Excluded observed weather variables and target-derived features because they would not be reliably available when predicting the following day's demand.

### 2. Model Training and Validation

Maintained the original chronological split:
- Training: 511 observations
- Validation: 109 observations
- Test: 111 observations (untouched)

Compared three configurations:

- **Model A:** LinearRegression with six original calendar features.
- **Model B:** LinearRegression with 14 engineered features.
- **Model C:** StandardScaler + Ridge(alpha=1.0) with the same engineered features.

The Ridge pipeline fits its scaler using training data only.

### 3. Experiment Results

| Model | Validation MAE | Validation RMSE | Mean Signed Error |
|---|---:|---:|---:|
| Model A — Original | 1,124.78 | 1,300.45 | +931.68 |
| Model B — Engineered | 711.21 | 932.96 | −51.72 |
| Model C — Ridge | **710.87** | **931.10** | **−42.68** |

### 4. Key Findings

**Feature engineering delivered the largest improvement.**

Model B reduced validation MAE by approximately 36.8% compared with Model A. One-hot encoding allowed independent weekday and season effects, while cyclical features represented repeating annual patterns.

**Ridge provided only a marginal additional improvement.**

At alpha=1.0, Ridge reduced MAE by approximately 0.34 rentals compared with unregularized Model B. This small difference does not establish a meaningful generalization advantage.

**Systematic underprediction was substantially reduced.**

Model A had a mean signed error of +931.68 rentals. Models B and C had mean signed errors much closer to zero, although both slightly overpredicted on average.

### 5. Testing and Validation

Completed:
- Leap-year date extraction assertions.
- One-hot encoding structural and semantic checks.
- Unit-circle identity verification.
- Year-boundary continuity test.
- Original baseline reproducibility.
- Three-model validation comparison.

The models were fitted on training observations and evaluated on validation observations only.

### 6. Limitations and Next Steps

- Validate engineered feature schema and missing-value checks.
- Verify the notebook executes successfully after restarting the kernel.
- Investigate error distributions and seasonal performance.
- Test alternative Ridge alpha values using appropriate validation procedures.
- Avoid using the historical binary `yr` feature directly for years beyond 2012.
- Consider timestamped weather forecasts for future models.
- Preserve the test set until final model evaluation.

**Conclusion:** Feature representation was substantially more influential than regularization in this experiment. Ridge achieved the lowest observed validation error, but its advantage over engineered linear regression was minimal.